In [11]:
import torch #創建tensors，儲存 raw data，以及一些有幫助的函數
import torch.nn as nn #使用Modual、Linear、Embedding classes以及一串有幫助的函數
import torch.nn.functional as F #接收softmax()，用來計算attention

from torch.optim import Adam #使用backpropagation你和神經網絡與資料
from torch.utils.data import TensorDataset, DataLoader#創造一個有很多training data的大型scale Transformer network

import lightning as L #自動最佳化與scaling，減少code

import re
from collections import Counter #用來計算文本每個字的出現次數

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # 選擇你的 txt 檔案上傳

In [21]:
def build_vocab(text, min_freq=1, special_tokens=["<PAD>", "<UNK>", "<EOS>"]):
    # 1. 清理文本並拆分單詞
    # 將文本轉為小寫，並用正則表達式提取單詞
    words = re.findall(r'\b\w+\b', text.lower())  # 轉小寫並提取單詞
    
    # 2. 計算詞頻
    word_freq = Counter(words)
    
    # 3. 過濾低頻詞（如果需要）
    vocab = special_tokens + [word for word, freq in word_freq.items() if freq >= min_freq]
    
    # 4. 轉換成詞典 (word -> id)
    token_to_id = {word: idx for idx, word in enumerate(vocab)}
    id_to_token = {idx: word for word, idx in token_to_id.items()}  # id -> word

    return token_to_id, id_to_token

# 讀取 apple.txt 檔案
with open("apples.txt", "r", encoding="utf-8") as file:
    text = file.read()

# 建立 token_to_id 和 id_to_token
token_to_id, id_to_token = build_vocab(text, min_freq=1)

# 輸出結果
# print("Token to ID:", token_to_id)
# print("ID to Token:", id_to_token)


In [24]:
# 將 token_to_id 的所有 ID 收集到一個列表中
model_input = [[token_to_id[token] for token in token_to_id]]

# 將列表轉換為 tensor
model_input_tensor = torch.tensor(model_input, dtype=torch.long)

# 將字符串轉換為對應的 ID
model_output_ids = [[token_to_id.get(token, token_to_id["<UNK>"]) for token in id_to_token]]

# 將 ID 列表轉換為 tensor
model_output_tensor = torch.tensor(model_output_ids, dtype=torch.long)

#將input和label送進TensorDataset()去創造dataset
dataset = TensorDataset(model_input_tensor, model_output_tensor)

#將dataset送入DataLoader()建立dataloadr
dataloader = DataLoader(dataset)

In [25]:
#以下code是用於計算前，把Position Encoding values加到tokens
class PositionEncoding(nn.Module):#定義新的class，PositionEncoding，繼承nn.Module

    def __init__(self, d_model=2, max_len=6):#dim of model，就是每個token，word Embedding的數量；max_len是Transformer的tokens的最大數量(包含inputs和outputs)
                       #這裡設d_model=2, max_len=6，但一般不會這麼小
        super().__init__() #呼叫nn.Module的__init__ method
        
        pe = torch.zeros(max_len, d_model)#創造一個Position Encoding values的矩陣，起始都為0s，命名為pe(Position Encoding)
                                          #pe的row與max_len一樣大，且col與d_model一樣大
        
        position = torch.arange(start=0, end=max_len, step=1).float().unsqueeze(1)#position是一個column matrix，表示每個token的位置(pos)
                                                                                 #用torch.arang()創造數列，再0與max_len之間，確定值是float，.unsqueeze(1)是轉換數列成一個column matrix
        embedding_index = torch.arange(start=0, end=d_model, step=2).float()#embedding_index是一個row matrix，代表對於每個word embedding，index i，i都乘2，
                                                                            #所以d_model = 2，embedding_index就是0，因為只有[0,1]，但步長又是2，所以只有單一的值；但d_model=6，輸出是[0,2,4]

        div_term = 1/torch.tensor(10000.0)**(embedding_index / d_model)

        #用position乘上div_term(有分之一)就是pos/(10000**(2i/d_model))
        pe[:, 0::2] = torch.sin(position * div_term)#這是把sin函數值放入pe矩陣，0是從第一個欄位，2是跳過下一個
        pe[:, 1::2] = torch.cos(position * div_term)#這是把cos函數值放入pe矩陣，1是從第二個欄位
          #所以這pe矩陣有2個cols，第一個是sin的值，第二個是cos的值
        
        self.register_buffer('pe', pe) #register_buffer()確定pe可以移到GPU，如果我們用一個GPU

    def forward(self, word_embeddings): #接受Word Embedding values
        return word_embeddings + self.pe[:word_embeddings.size(0), :]#加Position Encoding values到word_embeddings

#以上就是code Position Encoding

In [26]:
class Attention(nn.Module):
    def __init__(self, d_model=2):#需要知道每一個Token的Word Embedding的數量(d_model)，因為我們要定義Weight matrix的大小，才能創造Queries、Keys以及Values
                                  #因為d_model=2，表示每一個Weight matrix需要2 rows * 2cols
        super().__init__()
        self.W_q = nn.Linear(in_features=d_model, out_features=d_model, bias=False)#用nn.Linear()創造Weight matrix，並進行運算
                                                                                  #in_features定義Weight matrix的列數，所以用d_model設置它...；out_features定義Weight matrix的欄位數，所以也用d_model設置它
                                                                                  #在原始Transformer文獻，沒有刻意設置bias，所以這裡bias=False
             #W_q代表用來計算Query的weightings，但還沒有訓練過；W_q是Linear()的物件，所以不用儲存weightings，但之後它也會幫我們做計算
        self.W_k = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
             #W_k是Linear()的物件，代表用來計算Key的weightings
        self.W_v = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
             #W_v是Linear()的物件，代表用來計算Value的weightings

        self.row_dim = 0
        self.col_dim = 1 #為了讓我們能靈活地以順序或批次方式輸入訓練數據，我們創建了一些變數來追蹤哪些索引對應於行和列

    def forward(self, encodings_for_q, encodings_for_k, encodings_for_v, mask=None):#forward() method是要計算每一個token的Masked Self Attention values
                      #encodings_for_q, encodings_for_k, encodings_for_v: 這是為了用不同token encodings計算Query、Key以及Values，增加靈活性
                      #因為Encoder-Decoder Transformer有個東西叫Encoder-Decoder Attention，Keys和Value是在Encoder中，用encoded tokens計算，而Queries是在Decoder中，用edecoded tokens計算
                      #所以這encodings來自不同的sources，這讓我們更靈活於計算 Encoder-Decoder Attention
                      #並且，我們想要計算Masked Self-Attention，我們可以通過一個mask
        q = self.W_q(encodings_for_q)
        k = self.W_k(encodings_for_k)
        v = self.W_v(encodings_for_v)#前面已經用Linear()算出了每一個token的Query, key and Values

        #現在我們要計算Attention: Attention(Q,K,V) = SoftMax(QK**T/sqrt(d_k))V
        #先用torch.matmul將q和k的transpose相乘
        sims = torch.matmul(q, k.transpose(dim0=self.row_dim, dim1=self.col_dim))#這就是Query和Key的相似度，儲存在sims

        scaled_sims = sims / torch.tensor(k.size(self.col_dim)**0.5)#用每一個key的值的數量開根號，scale上面的相似度，會這樣是2017年原始Transformer文獻的做法，現在已是慣例，不過這在模型較大的情況比較有幫助

        
        #接下來我們要做的是將mask（如果有使用的話）加入縮放後的相似度中， Attention(Q,K,V) = SoftMax(QK**T/sqrt(d_k) +M)V
        if mask is not None:
            scaled_sims = scaled_sims.masked_fill(mask=mask, value=-1e9)#Mask是用於保護先前的tokens，以免被欺騙，以及檢查叫新的tokens
                                      #為了瞭解如何增加一個mask，使用masked_fill() method，它是一個T or F的矩陣，Trues相對應的Attention values是我們要忽略的，所以mask_fill()會替換這些值為-1e9，幾乎是-inf，然後False就替換成0
                                      #創造最終的mask，加入被scale的相似度: scaled_sims

        #接下來是，將scaled sims通過SoftMax()，去計算Attention
        attention_percents = F.softmax(scaled_sims, dim=self.col_dim) #用SoftMax()對scaled_相似度的輸出值，決定每個token對其他的影響比例

        attention_score = torch.matmul(attention_percents, v)#最後用torch.matmul()將attention_percents與Values相乘

        return attention_score

In [27]:
class DecoderOnlyTransformer(L.LightningModule):#跟PositionEncoding與Attention的class不同，這是繼乘LightningModule
                                                #這樣做，而不是讓每個類別都繼承自 LightningModule，可以讓我們利用 Lightning 提供的一切功能，同時避免多次繼承帶來的額外負擔
    def __init__(self, num_tokens=4, d_model=2, max_len=6):
        #這個__init__ method中，num_tokens是單字表中，tokens的數量，d_model是每個token，values的數量，max_len是inputs與outputs的最大長度
        super().__init__() #呼叫母體__init__的method

        self.we = nn.Embedding(num_embeddings=num_tokens, #創造Embedding()物件，取名為we(Word Embedding)，num_embeddings:要知道單字表中，token的數量，embedding_dim: 代表每一個token的值的數量
                               embedding_dim=d_model)
        self.pe = PositionEncoding(d_model=d_model, #使用前一個class創造的PositionEncoding()物件，取名為pe
                                   max_len=max_len)
        self.self_attention = Attention(d_model=d_model) #創造一個attention物件
        self.fc_layer = nn.Linear(in_features=d_model, out_features=num_tokens) #用nn.Linea()創造Fully Connected layer，nn.Linear()需要知道有多少個inputs進入fc layer，以及有多少個要輸出

        self.loss = nn.CrossEntropyLoss()#創要一個loss function，評估模型表現得好不好，這裡使用CrossEntropyLoss()，因為這模型有很多outputs，以及CrossEntropyLoss()也會幫我們使用SoftMax()

    #現在，我們將所有piece放進forward() method
    def forward(self, token_ids): #forward()會取得token id數量的array，作為Transformer的inputs
        #先把tokens轉換為Word Embedding values
        word_embeddings = self.we(token_ids)

        #再加入Position Encoding
        position_encoded = self.pe(word_embeddings)

        #然後我們創建一個mask，防止早期的標記在計算注意力時查看後期的標記
        mask = torch.tril(torch.ones((token_ids.size(dim=0), token_ids.size(dim=0))))
                         #torch.ones(): 製造一個都是1s的矩陣，例如:這裡有4個tokens，就是4*4的矩陣，其元素都是1
               #torch.tril(): tril(Tri-L)是指Lower triangle，也就是將都是1的4*4矩陣，其對角線與對角線以下的值都保留，其他上面的部分都變為0
        #所以，mask就是儲存一個對角線與對角線以下的值都是1，其它值都是0的Lower triangle matrix的變數

        mask = mask == 0 #表示將矩陣中的0變成True，1變成False；最後，用4個tokwn ids建的mask，會被用於masked-attention，當有了mask，就開始計算attention

        self_attention_values = self.self_attention(position_encoded, # calculate for Queries
                                                    position_encoded, # calculate for Keys
                                                    position_encoded, # calculate for Values
                                                    mask=mask) # 通過mask，防止早期的標記在計算注意力時查看後期的標記
        #Note: 因為Queries、Keys以及Values都是由一樣的token embeddings，所以要通過3次一樣的Position Emcoding的集合，依次是Queries、Keys以及Values

        #加入Residual Connections
        residual_connection_values = position_encoded + self_attention_values

        #最後，通過Fully Connected Layer
        fc_layer_output = self.fc_layer(residual_connection_values)
                                        
        return fc_layer_output #在CrossEntropyLoss()會使用SoftMax()，所以我們要從Fully Connected Layer輸出output

        #以上我們已經寫完Decoding-Only Transformer ，接下來是要訓練它
    def configure_optimizers(self):
        return Adam(self.parameters(), lr=0.1)#我們將模型中所有我們想要訓練的weightings和biases傳遞給 Adam，這包含了所有的weightings和biases，lr=0.1: 可以使一些特定模型訓練的非常快(預設是0.001，所以可能會比較普遍)

    def training_step(self, batch, batch_idx):#取一部分的training data與那部分的index
        input_tokens, labels = batch #區分training data為inputs和labels
        output = self.forward(input_tokens[0]) #將input_tokens放入forward()，算出output
        loss = self.loss(output, labels[0]) #用loss function，比較Transformer的outputs與已知的labels；且loss function會幫我們再用SoftMax()

        return loss 


In [28]:
import torch
import torch.nn.functional as F

# 假設這是你的 token_to_id 和 id_to_token 字典

# 假設你已經建立並訓練了模型
model = DecoderOnlyTransformer(num_tokens=len(token_to_id), d_model=50, max_len=len(token_to_id))

# 將 token_to_id 的所有 ID 收集到一個列表中
model_input = [token_to_id[token] for token in token_to_id]
model_input_tensor = torch.tensor(model_input, dtype=torch.long)

input_length = model_input_tensor.size(dim=0)

predicted_ids = torch.tensor([], dtype=torch.long)  # 初始化空的預測 ID tensor
predicted_ids = torch.cat((predicted_ids, model_input_tensor))  # 加入初始輸入

max_length = len(token_to_id)
for i in range(input_length, max_length):
    predictions = model(predicted_ids.unsqueeze(0))  # 確保輸入形狀正確 (batch_size, seq_len)

    # 取得最後一個 token 的預測
    probs = F.softmax(predictions[:, -1, :], dim=-1)  # 取得最後一個 token 的預測機率
    predicted_id = torch.multinomial(probs, num_samples=1)  # 使用多項式採樣
    predicted_ids = torch.cat((predicted_ids, predicted_id.squeeze()))  # 添加預測的 token

    if predicted_id.item() == token_to_id["<EOS>"]:  # 如果預測的 token 是 <EOS>
        break

# 輸出預測的 tokens
print("Predicted Tokens:\n")
for id in predicted_ids:
    print("\t", id_to_token[id.item()])


Predicted Tokens:

	 <PAD>
	 <UNK>
	 <EOS>
	 1
	 origin
	 and
	 history
	 of
	 apples
	 malus
	 domestica
	 are
	 among
	 the
	 most
	 popular
	 fruits
	 worldwide
	 belonging
	 to
	 rosaceae
	 family
	 genus
	 can
	 be
	 traced
	 back
	 central
	 asia
	 particularly
	 modern
	 day
	 kazakhstan
	 wild
	 ancestor
	 apple
	 sieversii
	 still
	 grows
	 in
	 tian
	 shan
	 mountains
	 were
	 spread
	 along
	 silk
	 road
	 europe
	 where
	 they
	 cultivated
	 improved
	 over
	 centuries
	 3000
	 bce
	 ancient
	 egyptians
	 began
	 cultivating
	 greek
	 roman
	 empires
	 introduced
	 breeding
	 techniques
	 throughout
	 17th
	 century
	 european
	 colonists
	 brought
	 north
	 america
	 making
	 them
	 a
	 staple
	 fruit
	 19th
	 john
	 chapman
	 also
	 known
	 as
	 johnny
	 appleseed
	 played
	 key
	 role
	 spreading
	 orchards
	 across
	 united
	 states
	 2
	 major
	 producing
	 regions
	 widely
	 grown
	 temperate
	 climates
	 top
	 countries
	 include
	 china
	 world
	 s
	 largest
	 produ

In [29]:
#要訓練模型的code就那麼簡單
trainer = L.Trainer(max_epochs=3) #創造一個Lighting Trainer，並告訴他只要做30次epochs，這對我們簡單的model和dataset就夠了
trainer.fit(model, train_dataloaders=dataloader)#使用fit()，將model與dataloader通過trainer

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name           | Type             | Params | Mode 
------------------------------------------------------------
0 | we             | Embedding        | 17.1 K | train
1 | pe             | PositionEncoding | 0      | train
2 | self_attention | Attention        | 7.5 K  | train
3 | fc_layer       | Linear           | 17.4 K | train
4 | loss           | CrossEntropyLoss | 0      | train
------------------------------------------------------------
42.0 K    Trainable params
0         Non-trainable params
42.0 K    Total params
0.168     Total estimated model params size (MB)
8         Modules in train mode
0         Modules in eval mode
C:\Python310\lib\site-packages\lightning\pytorch\loops\fit_loop.py:310: The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for 

Training: |                                                                                      | 0/? [00:00<…

`Trainer.fit` stopped: `max_epochs=3` reached.


In [34]:
import torch
import torch.nn.functional as F

# 假設這是你的 token_to_id 和 id_to_token 字典

# 假設你已經建立並訓練了模型
model = DecoderOnlyTransformer(num_tokens=len(token_to_id), d_model=50, max_len=len(token_to_id))

# 將 token_to_id 的所有 ID 收集到一個列表中
model_input = [token_to_id[token] for token in token_to_id]
model_input_tensor = torch.tensor(model_input, dtype=torch.long)

input_length = model_input_tensor.size(dim=0)

predicted_ids = torch.tensor([], dtype=torch.long)  # 初始化空的預測 ID tensor
predicted_ids = torch.cat((predicted_ids, model_input_tensor))  # 加入初始輸入

max_length = len(token_to_id)
for i in range(input_length, max_length):
    predictions = model(predicted_ids.unsqueeze(0))  # 確保輸入形狀正確 (batch_size, seq_len)

    # 取得最後一個 token 的預測
    probs = F.softmax(predictions[:, -1, :], dim=-1)  # 取得最後一個 token 的預測機率
    predicted_id = torch.multinomial(probs, num_samples=1)  # 使用多項式採樣
    predicted_ids = torch.cat((predicted_ids, predicted_id.squeeze()))  # 添加預測的 token

    if predicted_id.item() == token_to_id["<EOS>"]:  # 如果預測的 token 是 <EOS>
        break

# 輸出預測的 tokens
print("Predicted Tokens:\n")
for id in predicted_ids:
    print("\t", id_to_token[id.item()])


Predicted Tokens:

	 <PAD>
	 <UNK>
	 <EOS>
	 1
	 origin
	 and
	 history
	 of
	 apples
	 malus
	 domestica
	 are
	 among
	 the
	 most
	 popular
	 fruits
	 worldwide
	 belonging
	 to
	 rosaceae
	 family
	 genus
	 can
	 be
	 traced
	 back
	 central
	 asia
	 particularly
	 modern
	 day
	 kazakhstan
	 wild
	 ancestor
	 apple
	 sieversii
	 still
	 grows
	 in
	 tian
	 shan
	 mountains
	 were
	 spread
	 along
	 silk
	 road
	 europe
	 where
	 they
	 cultivated
	 improved
	 over
	 centuries
	 3000
	 bce
	 ancient
	 egyptians
	 began
	 cultivating
	 greek
	 roman
	 empires
	 introduced
	 breeding
	 techniques
	 throughout
	 17th
	 century
	 european
	 colonists
	 brought
	 north
	 america
	 making
	 them
	 a
	 staple
	 fruit
	 19th
	 john
	 chapman
	 also
	 known
	 as
	 johnny
	 appleseed
	 played
	 key
	 role
	 spreading
	 orchards
	 across
	 united
	 states
	 2
	 major
	 producing
	 regions
	 widely
	 grown
	 temperate
	 climates
	 top
	 countries
	 include
	 china
	 world
	 s
	 largest
	 produ